# 1.1. DataPrep: DILI labels
# 1.1. 数据准备：DILI 标签

### 📘 Overview
### 📘 概述

This notebook integrates clinical DILI annotations (from resources such as DILIrank and LiverTox) with compound-level metadata (e.g., SMILES, molecular weight). It assigns consensus DILI labels for all compounds and generates a unified dataset.
这个notebook整合了临床DILI注释（来自DILIrank和LiverTox等资源）与化合物级别的元数据（例如SMILES、分子量）。它为所有化合物分配一致的DILI标签，并生成统一的数据集。

**Inputs**
**输入数据**
- [DILIrank](https://www.fda.gov/science-research/bioinformatics-tools/liver-toxicity-knowledge-base-ltkb)  annotations based on FDA-approved labels
- [DILIrank](https://www.fda.gov/science-research/bioinformatics-tools/liver-toxicity-knowledge-base-ltkb)：基于FDA批准标签的注释
- [LiverTox](https://www.ncbi.nlm.nih.gov/books/NBK548049) annotations of DILI mechanisms and likelihood scores
- [LiverTox](https://www.ncbi.nlm.nih.gov/books/NBK548049)：DILI机制和可能性评分的注释
- CHEMBL and selleckchem for compound metadata such as SMILES
- CHEMBL和selleckchem：用于获取化合物元数据，如SMILES

**Output**  
**输出**  
A consolidated CSV file containing:  
一个整合的CSV文件，包含：  
- Compound metadata  
- 化合物元数据  
- DILI annotations from each source  
- 来自每个数据源的DILI注释  
- Final consensus DILI label
- 最终一致的DILI标签

In [18]:
%%capture

!pip install PyPDF2

In [19]:
import os
import requests

import numpy as np
import pandas as pd

import tarfile
import PyPDF2

import dilimap as dmap

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
dmap.logging.print_version()

Running dilimap 0.2.dev20+gc285bd842 (python 3.10.19) on 2025-11-20 01:12.


## 1. Download relevant files to S3
## 1. 下载相关文件到S3

这一步会从FDA等官方来源下载DILI相关的数据文件，包括DILIrank、DILIst、LiverTox和Selleckchem的数据。

In [22]:
filedir = '../data/'

In [ ]:
# ⚠️ Set to True when updating DILIrank / DILIst / LiverTox files from FDA sources
# ⚠️ 当需要从FDA来源更新DILIrank / DILIst / LiverTox文件时，将此设置为True
# 注意：这个代码块默认是False，因为数据已经下载过了
if False:
    os.makedirs(filedir, exist_ok=True)

    # DILIrank (https://www.sciencedirect.com/science/article/abs/pii/S1359644616300411)
    !curl -L 'https://www.fda.gov/media/113052/download' -o '../data/DILIrank.xlsx'
    pd.read_excel(
        f'{filedir}DILIrank.xlsx', engine='openpyxl', header=1, index_col=0
    ).to_csv(f'{filedir}DILIrank.csv')

    # DILIst (https://www.sciencedirect.com/science/article/pii/S1359644619303824)
    !curl -L 'https://www.fda.gov/media/160597/download' -o '../data/DILIst.xlsx'
    pd.read_excel(
        f'{filedir}DILIst.xlsx', engine='openpyxl', header=0, index_col=0
    ).to_csv(f'{filedir}DILIst.csv')

    # LiverTox (https://www.ncbi.nlm.nih.gov/books/NBK547852/)
    !curl -L 'https://www.ncbi.nlm.nih.gov/books/NBK571102/bin/masterlist01-25.xlsx' -o '../data/LiverTox.xlsx'
    pd.read_excel(
        f'{filedir}LiverTox.xlsx', engine='openpyxl', header=1, index_col=0
    ).dropna(how='all')[:1644].to_csv(f'{filedir}LiverTox.csv')

    # Selleckchem (https://www.selleckchem.com/screening/chemical-library.html)
    !curl 'https://file.selleckchem.com/downloads/library/20250519-L1700-Bioactive-Compound-Library-I-96-well.xlsx' -o '../data/Selleckchem.xlsx'
    df_selleckchem = pd.read_excel(
        f'{filedir}Selleckchem.xlsx',
        sheet_name='L1700-Bioactive-9934 cpds',
        index_col=0,
    ).to_csv(f'{filedir}Selleckchem.csv')

    # Push to S3
    for filename in ['DILIst', 'DILIrank', 'LiverTox', 'Selleckchem']:
        dmap.s3.write(
            f'{filedir}{filename}.csv',
            filename=f'{filename}.csv',
            package_name='public/data',
        )

## 2. DILI labels (DILIrank, DILIst & LiverTox)
## 2. DILI标签（DILIrank、DILIst和LiverTox）

这一步整合来自三个主要数据源的DILI标签：
- **DILIrank**：FDA的肝毒性知识库，基于药物标签信息
- **DILIst**：DILI标准列表，包含分类信息
- **LiverTox**：NIH的肝毒性数据库，包含机制和可能性评分

In [ ]:
def lower(index):
    # 辅助函数：将索引转换为小写
    return index.str.lower()


def drop_duplicated_index(df, keep='first'):
    # 辅助函数：删除重复的索引（保留第一个）
    return df[~df.index.str.lower().duplicated(keep='first')]


def capitalize_if_lower(index):
    # 辅助函数：如果字符串是小写，则首字母大写
    return [s.capitalize() if s.islower() else s for s in index]

In [25]:
df_DILIrank = dmap.s3.read('DILIrank.csv')
df_DILIst = dmap.s3.read('DILIst.csv')
df_livertox = dmap.s3.read('LiverTox.csv')
df_cmpd_list = dmap.s3.read('compound_list_for_industry_benchmark.csv')

Package: s3://dilimap/public/data. Top hash: 155a2b3b63
Package: s3://dilimap/public/data. Top hash: 155a2b3b63
Package: s3://dilimap/public/data. Top hash: 155a2b3b63
Package: s3://dilimap/public/data. Top hash: 155a2b3b63


In [ ]:
df_DILIrank = df_DILIrank.reset_index()
# 重置索引，准备标准化化合物名称

df_DILIrank.index = df_DILIrank['Compound Name'].str.capitalize().str.rstrip(' ')
df_DILIst.index = df_DILIst['CompoundName'].str.capitalize().str.rstrip(' ')
df_livertox.index = df_livertox['Ingredient'].str.capitalize().str.rstrip(' ')
# 标准化化合物名称：首字母大写，去除末尾空格

df_DILIrank = df_DILIrank.drop_duplicates(subset='Compound Name')
df_DILIst = df_DILIst.drop_duplicates(subset='CompoundName')
df_livertox = df_livertox.drop_duplicates(subset='Ingredient')
# 删除重复的化合物（基于化合物名称列）

In [ ]:
df_DILIst['source'] = np.nan
# 标记DILIst数据集中每个化合物的来源（DILIst数据集包含来自多个研究的数据）
df_DILIst.iloc[909:, df_DILIst.columns.get_loc('source')] = 'LiverTox'  # 来自LiverTox数据库
df_DILIst.iloc[952:, df_DILIst.columns.get_loc('source')] = 'Suzuki'   # 来自Suzuki研究
df_DILIst.iloc[1014:, df_DILIst.columns.get_loc('source')] = 'Greene'   # 来自Greene研究
df_DILIst.iloc[1107:, df_DILIst.columns.get_loc('source')] = 'Zhu'      # 来自Zhu研究

df_DILIst.iloc[:909, df_DILIst.columns.get_loc('DILIst Classification ')] = np.nan
# 前909个化合物没有DILIst分类，设为NaN

/var/folders/fh/r5412d0d6tl262nd_38k08d80000gp/T/ipykernel_16212/328512810.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'LiverTox' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_DILIst.iloc[909:, df_DILIst.columns.get_loc('source')] = 'LiverTox'


In [28]:
df_DILIrank.index = capitalize_if_lower(df_DILIrank.index)
df_DILIst.index = capitalize_if_lower(df_DILIst.index)
df_livertox.index = capitalize_if_lower(df_livertox.index)
df_cmpd_list.index = capitalize_if_lower(df_cmpd_list.index)

In [29]:
df_DILI = pd.merge(
    df_DILIrank, df_DILIst, left_index=True, right_index=True, how='outer'
)

In [ ]:
df_DILI['severity_class'] = df_DILI['Severity Class']
# 提取和标准化DILI相关字段：严重程度等级
df_DILI['label_section'] = df_DILI['Label Section']
# 标签部分
df_DILI['DILIrank'] = df_DILI['vDILIConcern'].str.split('v').str[-1]
# 提取DILIrank分类（去掉'v'前缀）
df_DILI['DILIst'] = np.where(
    df_DILI['DILIst Classification '] == 1,
    'DILI',
    # 分类为1表示有DILI
    np.where(df_DILI['DILIst Classification '] == 0, 'No-DILI', np.nan),
    # 分类为0表示无DILI
)
df_DILI['roa'] = df_DILI['Routs of Administration ']
# 给药途径
df_DILI['source'] = df_DILI['source'].fillna('DILIrank')
# 默认来源为DILIrank

# Combine all required compound indices
# 合并所有需要的化合物索引（包括基准测试列表、LiverTox和手动添加的化合物）
required_cmpds = df_cmpd_list.index.union(df_livertox.index).union(
    [
        'Asunaprevir',
        'FIRU',
        'Lomitapide',
        'TAK-875',
        'AKN-028',
        'Evobrutinib',
        'BMS-986142',
        'Ibrutinib',
        'Orelabrutinib',
        'Remibrutinib',
        'Rilzabrutinib',
        'Ruxolitinib',
        'Tofacitinib',
        'Upadacitinib',
    ]
)

# Add any missing compounds to df_DILI with NaN rows
# 为缺失的化合物添加空行
df_DILI = df_DILI.reindex(df_DILI.index.union(required_cmpds))

df_DILI['compound_name'] = df_DILI.index

df_DILI = df_DILI[
    [
        'LTKBID',
        # LiverTox知识库ID
        'compound_name',
        # 化合物名称
        'DILIrank',
        # DILIrank分类
        'label_section',
        # 标签部分
        'severity_class',
        # 严重程度等级
        'DILIst',
        # DILIst分类
        'roa',
        # 给药途径
        'source',
        # 数据来源
    ]
]
df_DILI = df_DILI.sort_values(['source', 'compound_name'])
# 按来源和化合物名称排序

In [31]:
df_DILI['livertox_score'] = df_DILI.index.map(df_livertox['Likelihood Score'])
df_DILI['livertox_primary_classification'] = df_DILI.index.map(
    df_livertox['Primary Classification']
)
df_DILI['livertox_secondary_classification'] = df_DILI.index.map(
    df_livertox['Secondary Classification']
)

In [32]:
df_DILI = drop_duplicated_index(df_DILI)

## 3. 药物信息（CHEMBL）

从CHEMBL数据库获取化合物的详细信息，包括SMILES结构、分子量等化学信息。

In [33]:
df_chembl = dmap.clients.chembl(df_DILI.index)

ValueError: The truth value of a Index is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
len(df_DILI), len(df_chembl['smiles']), len(df_chembl['smiles'].dropna())

(2475, 2145, 1806)

In [ ]:
df_chembl.reset_index(inplace=True)
df_chembl.set_index('molecule_name', inplace=True)

df_chembl.index = lower(df_chembl.index)
df_chembl = drop_duplicated_index(df_chembl)

In [ ]:
for k in df_chembl.columns:
    df_DILI[k] = lower(df_DILI.index).map(df_chembl[k])

In [ ]:
drug_warnings = dmap.clients.drug_warnings(chembl_id=df_DILI['molecule_chembl_id'])

In [ ]:
## Since the client slightly lags behind on one drug
## 由于客户端对某个药物的数据略有滞后，需要手动添加

# CHEMBL623（某个特定药物）的警告信息可能缺失，需要手动补充
if 'CHEMBL623' not in drug_warnings.index:
    drug_warnings.loc['CHEMBL623'] = {
        'efo_id': 'EFO:0004228',
        'efo_id_for_warning_class': 'EFO:0011052',
        'efo_term': 'drug-induced liver injury',
        'parent_molecule_chembl_id': 'CHEMBL623',
        'warning_class': 'hepatotoxicity',
        'warning_country': 'Canada: 2003',
        'warning_description': 'Adverse hepatic events',
        'warning_type': 'Withdrawn',
    }

In [ ]:
drug_warnings.index = lower(drug_warnings.index)

for key in [
    'warning_class',
    'warning_country',
    'warning_description',
    'warning_type',
    'warning_year',
]:
    if key in drug_warnings.columns:
        df_DILI[key] = lower(df_DILI['molecule_chembl_id']).map(drug_warnings[key])

In [ ]:
df_DILI['withdrawn_reason'] = np.where(
    df_DILI['warning_type'] == 'Withdrawn', df_DILI['warning_class'], np.nan
)

## 4. 药物靶点和溶解度信息（Selleckchem）

从Selleckchem数据库获取化合物的靶点、通路信息和DMSO溶解度数据。

In [ ]:
df_selleckchem = dmap.s3.read('Selleckchem.csv')

Package: s3://dilimap/public/data. Top hash: e13a57ac61


In [ ]:
df_selleckchem.index = lower(df_selleckchem['Name'])
df_selleckchem.index = df_selleckchem.index.str.split(r' \(').str[0].str.rstrip(' ')
df_selleckchem = drop_duplicated_index(df_selleckchem)

In [ ]:
for k in ['Target', 'Pathway', 'Information']:
    df_DILI[k.lower()] = lower(df_DILI.index).map(df_selleckchem[k])

df_DILI['CAS_number'] = lower(df_DILI.index).map(df_selleckchem['CAS Number'])
df_DILI['DMSO_mM_solubility'] = lower(df_DILI.index).map(
    df_selleckchem['DMSO (mM)Max Solubility']
)

## 5. 一致的DILI标签

根据DILIrank和LiverTox的评分，为每个化合物分配最终的一致DILI标签。标签分为多个级别：
- **DILI (withdrawn)**：因肝毒性而撤市
- **DILI (known)**：已知DILI（高置信度）
- **DILI (likely)**：可能DILI
- **DILI (few cases)**：少数病例报告
- **No DILI**：无DILI
- **No DILI (unlikely)**：不太可能有DILI

In [ ]:
for k in ['DILIrank', 'livertox_score', 'label_section', 'withdrawn_reason']:
    if k in df_DILI.columns:
        df_DILI[k] = df_DILI[k].replace(np.nan, '')

In [ ]:
# --- Define consensus DILI labels from DILIrank and LiverTox ---
# --- 根据DILIrank和LiverTox定义一致的DILI标签 ---

# Define terms
# 定义判断条件
is_Most = df_DILI['DILIrank'].str.startswith('Most')
# DILIrank分类为"Most-DILI"
is_No = df_DILI['DILIrank'].str.startswith('No')
# DILIrank分类为"No-DILI"
is_Withdrawn = df_DILI['label_section'].eq('Withdrawn')
# 标签部分为"撤市"
is_Withdrawn |= (
    df_DILI['withdrawn_reason'].str.lower().str.contains('liver|hepa', na=False)
)
# 撤市原因包含肝相关
is_A = df_DILI['livertox_score'].str.startswith('A')
# LiverTox评分为A（明确DILI）
is_AB = df_DILI['livertox_score'].str.startswith(('A', 'B'))
# LiverTox评分为A或B
is_E = df_DILI['livertox_score'].str.startswith('E')
# LiverTox评分为E（不太可能DILI）

# Withdrawn due to hepatotoxicity
# 因肝毒性而撤市：Most-DILIrank且撤市
Withdrawn = is_Most & is_Withdrawn

# Known DILI: High-confidence score A
# 已知DILI：高置信度评分A
# also consider if not explicitly listed in DILIrank
# 如果DILIrank中没有明确列出，但LiverTox评分为A，也考虑为已知DILI
Most_and_A = (
    is_Most | df_DILI['DILIrank'].eq('')
) & is_A

# Likely DILI: Most-DILIrank or A/B score
# 可能DILI：Most-DILIrank或A/B评分
Most_or_AB = is_Most | is_AB

# Few case reports or ambiguous evidence
# 少数病例报告或模糊证据：LiverTox评分为C或D
C_or_D = df_DILI['livertox_score'].str.startswith(('C', 'D'))

# No DILI
# 无DILI：多种条件组合
No_and_E = (
    (
        df_DILI['DILIrank'].str.startswith(('No', 'Ambiguous'))
        # DILIrank为No或Ambiguous
        & (is_E | (df_DILI['severity_class'] <= 1))
        # 且LiverTox为E或严重程度≤1
    )
    | (is_No & df_DILI['livertox_score'].eq(''))
    # 或DILIrank为No且无LiverTox评分
    | (df_DILI['DILIrank'].eq('') & is_E)
    # 或无DILIrank但LiverTox为E
    | (
        df_DILI['DILIrank'].eq('')
        & df_DILI['livertox_score'].eq('')
        & df_DILI['DILIst'].eq('No-DILI')
        # 或无DILIrank和LiverTox但DILIst为No-DILI
    )
)


# --- Manual additions ---
# --- 手动添加的化合物分类 ---

# 手动标记为撤市的化合物
Withdrawn |= df_DILI['compound_name'].isin(
    ['AKN-028', 'TAK-875', 'Evobrutinib', 'BMS-986142', 'Orelabrutinib']
)

# 手动标记为可能DILI的化合物
Most_or_AB |= df_DILI['compound_name'].isin(
    [
        'Phenylbutazone',
        'Flucloxacillin',
        'Salsalate',
        'TAK-875',
        'AKN-028',
        'Evobrutinib',
        'BMS-986142',
    ]
)

# 手动标记为少数病例的化合物
C_or_D |= df_DILI['compound_name'].isin(
    [
        'Mesalazine',
        'Aceclofenac',
        'Glibenclamide',
        'Mycophenolate mofetil',
        'Valdecoxib',
        'Tenoxicam',
        'Ibrutinib',
        'Vismodegib',
    ]
)

# 手动标记为无DILI的化合物
No_and_E |= df_DILI['compound_name'].isin(['FIRU', 'Upadacitinib'])

# --- Assign multi-level DILI labels ---
# --- 分配多级DILI标签 ---

# 根据条件优先级分配标签（优先级从高到低）
df_DILI['DILI_label'] = np.select(
    [Withdrawn, Most_and_A, Most_or_AB, No_and_E, C_or_D, is_E],
    [
        'DILI (withdrawn)',
        # 因肝毒性撤市
        'DILI (known)',
        # 已知DILI
        'DILI (likely)',
        # 可能DILI
        'No DILI',
        # 无DILI
        'DILI (few cases)',
        # 少数病例
        'No DILI (unlikely)',
        # 不太可能DILI
    ],
    default='',
    # 默认空值
)

# Binary DILI vs No-DILI label
# 二分类标签：DILI vs No-DILI
df_DILI['DILI_label_binary'] = np.where(
    Withdrawn | Most_and_A | Most_or_AB, 'DILI', np.where(No_and_E, 'No DILI', '')
    # 如果有DILI，如果明确无DILI，否则为空
)

# Additional section label from what's in paranthesis (for grouping if needed)
# 提取标签括号中的部分（用于分组）
df_DILI['DILI_label_section'] = (
    df_DILI['DILI_label'].str.extract(r'\(([^)]+)\)', expand=False).str.capitalize()
)
# 如果没有括号（如"No DILI"），则使用完整标签
df_DILI['DILI_label_section'] = df_DILI['DILI_label_section'].fillna(
    df_DILI['DILI_label']
)

In [ ]:
df_DILI['DILI_label'].value_counts()

DILI_label
No DILI               1009
                       697
DILI (few cases)       328
DILI (likely)          234
DILI (known)           113
DILI (withdrawn)        62
No DILI (unlikely)      32
Name: count, dtype: int64

## 6. Mechanisms of DILI
## 6. DILI机制

Optional — not used for model training, but to support faster interpretation.
可选步骤 — 不用于模型训练，但有助于快速解释结果。

这一步从LiverTox的PDF文档中提取DILI的机制信息，包括：
- 肝损伤机制（如肝炎、坏死、胆汁淤积等）
- 是否为特发性DILI（idiosyncratic DILI）
- 机制摘要

In [ ]:
# Download livertox book
# 下载LiverTox书籍（PDF格式）
# 这个文件包含每个药物的详细机制描述
if not os.path.exists(f'{filedir}livertox_NBK547852/'):
    response = requests.get(
        'https://ftp.ncbi.nlm.nih.gov/pub/litarch/29/31/livertox_NBK547852.tar.gz',
        stream=True,
    )
    file = tarfile.open(fileobj=response.raw, mode='r|gz')
    file.extractall(path='../data/')

    file.close()

    # remove all non-pdf files
    # Fix: Convert index to list to avoid "ambiguous truth value" error
    dili_index_list = df_DILI.index.tolist()
    for root, dirs, files in os.walk('../data/livertox_NBK547852/'):
        for file in files:
            if not file.lower().endswith('.pdf') and file[:-4] not in dili_index_list:
                file_path = os.path.join(root, file)
                os.remove(file_path)

In [ ]:
import os
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

kw_list = [
    'Mechanism of Injury\n',
    'Mechanism of Liver Injury\n',
    'Mechanism of injury\n',
    'Mechanism of Hepatotoxicity\n',
    'Mechanism of Injur y\n',  # typo in Zonisamide
    # Zonisamide中的拼写错误
]
# 机制部分的关键词列表（用于在PDF中定位机制描述）


def extract_info(cmpd):
    """从PDF文件中提取化合物的机制信息"""
    filepath = f'{filedir}livertox_NBK547852/{cmpd}.pdf'
    if not os.path.exists(filepath):
        return cmpd, np.nan, np.nan, np.nan, df_DILI.loc[cmpd, 'DILI_label']

    try:
        reader = PyPDF2.PdfReader(filepath)
        # 读取PDF文件（只读前10页，因为机制信息通常在前面）
        text = ''.join([page.extract_text() or '' for page in reader.pages[:10]])

        # Identify the mechanism section keyword
        # 识别机制部分的关键词
        kw = next((k for k in kw_list if k in text), None)
        if kw is None:
            return cmpd, np.nan, np.nan, np.nan, df_DILI.loc[cmpd, 'DILI_label']

        # Extract mechanism
        # 提取机制描述（从机制关键词到"结果和管理"部分之间）
        mech = text.split(kw)[-1].split('Outcome and Management')[0].replace('\n', '')

        # Extract overview text
        # 提取概述文本
        info = np.nan
        if 'OVERVIEW\n' in text:
            info = text.split('OVERVIEW\n')[1].split(kw)[0]
            info = (
                info.replace('Introduction\n', 'INTRODUCTION ')
                .replace('Background\n', ' BACKGROUND ')
                .replace('Hepatotoxicity\n', ' HEPATOTOXICITY ')
                .replace('\n', '')
            )

        # Extract update timestamp
        # 提取更新时间戳
        updated = text.split('[Updated ')[1].split(']')[0].replace('\n', '')

        return cmpd, mech, info, updated, np.nan

    except Exception as e:
        print(f'Error processing {cmpd}: {e}')
        # 处理 {cmpd} 时出错
        return cmpd, np.nan, np.nan, np.nan, df_DILI.loc[cmpd, 'DILI_label']


# Run in parallel
# 并行处理所有化合物（使用8个工作线程）
results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(extract_info, cmpd) for cmpd in df_DILI.index]
    for future in as_completed(futures):
        results.append(future.result())

# Update df_DILI with results
# 用提取的结果更新df_DILI
for cmpd, mech, info, updated, warn_label in results:
    if not pd.isna(mech):
        df_DILI.loc[cmpd, 'livertox_mechanism'] = mech
        # 机制描述
        df_DILI.loc[cmpd, 'livertox_information'] = info
        # 概述信息
        df_DILI.loc[cmpd, 'livertox_updated'] = updated
        # 更新时间

# Final cleanup
# 最终清理：将双空格替换为单空格
df_DILI['livertox_mechanism'] = df_DILI['livertox_mechanism'].str.replace(
    '  ', ' ', regex=False
)
df_DILI['livertox_information'] = df_DILI['livertox_information'].str.replace(
    '  ', ' ', regex=False
)

In [ ]:
idx_moa = df_DILI['livertox_mechanism'].astype(str).str.contains('idiosyncratic')
idx_moa &= ~df_DILI['livertox_mechanism'].astype(str).str.contains('not idiosyncratic')
idx_moa &= df_DILI['livertox_score'].str.startswith('C') | df_DILI[
    'livertox_score'
].str.startswith('D')

idx = {}
for kw in [
    'clearly',
    'likely',
    'probably',
    'suggest',
    'may be',
    'whether this injury is',
    'has features of',
    'various',
    'cases',
    'instances of',
]:
    idx[kw] = (
        df_DILI['livertox_mechanism']
        .str.split('idiosyncratic')
        .str[0]
        .str[-30:]
        .str.lower()
        .str.contains(kw)
    )

df_DILI['livertox_iDILI'] = np.where(
    idx_moa & idx['clearly'],
    'idiosyncratic',
    np.where(
        idx_moa & (idx['likely'] | idx['probably']),
        'likely idiosyncratic',
        np.where(
            idx_moa
            & (
                idx['may be']
                | idx['suggest']
                | idx['whether this injury is']
                | idx['has features of']
            ),
            'possibly idiosyncratic',
            np.where(
                idx_moa & (idx['various'] | idx['cases'] | idx['instances of']),
                'idiosyncratic cases',
                np.where(idx_moa, 'idiosyncratic', ''),
            ),
        ),
    ),
)

idx_dili = df_DILI['livertox_information'].astype(str).str.contains('idiosyncratic')
idx_dili &= ~df_DILI['livertox_information'].astype(str).str.contains(
    'not idiosyncratic'
)
idx_dili &= df_DILI['livertox_score'].str.startswith('C') | df_DILI[
    'livertox_score'
].str.startswith('D')

idx = {}
for kw in [
    'clearly',
    'most common causes',
    'well known cause of',
    'well established cause',
    'likely',
    'probably',
    'suggest',
    'may be',
    'whether this injury is',
    'has features of',
    'various',
    'cases',
    'instances of',
    'the few',
]:
    idx[kw] = (
        df_DILI['livertox_information']
        .str.split('idiosyncratic')
        .str[0]
        .str[-45:]
        .str.lower()
        .str.contains(kw)
    )

df_DILI['livertox_iDILI'] = np.where(
    idx_moa,
    df_DILI['livertox_iDILI'],
    np.where(
        idx_dili
        & (
            idx['clearly']
            | idx['most common causes']
            | idx['well known cause of']
            | idx['well established cause']
        ),
        'idiosyncratic',
        np.where(
            idx_dili & (idx['likely'] | idx['probably']),
            'likely iDILI',
            np.where(
                idx_dili
                & (
                    idx['may be']
                    | idx['suggest']
                    | idx['whether this injury is']
                    | idx['has features of']
                ),
                'possibly idiosyncratic',
                np.where(
                    idx_dili
                    & (
                        idx['various']
                        | idx['cases']
                        | idx['instances of']
                        | idx['the few']
                    ),
                    'idiosyncratic cases',
                    np.where(idx_dili, 'idiosyncratic cases', ''),
                ),
            ),
        ),
    ),
)

In [ ]:
df_tmp = df_DILI.copy()

moas = [
    'Hepatitis',
    'Necrosis',
    'Mitochondrial dysfunction',
    'CYP',
    'Oxidative stress',
    'Cholestasis/Biliary',
    'Sinusoidal',
    'Steatosis',
    'Fibrosis',
    'Cirrhosis',
    'Lactic acidosis',
    'Encephalopathy',
    'Immune-mediated',
    'Hypersensitivity',
    'Hypersensitivity Syndrome',
]

is_DILI = df_tmp['DILI_label_section'].isin(
    ['Withdrawn', 'Known', 'Likely', 'Few cases']
)
livertox_text = (
    df_tmp['livertox_information'].astype(str)
    + df_tmp['livertox_mechanism'].astype(str)
).str.lower()

for moa in moas:
    df_tmp[f'livertox_{moa}'] = livertox_text.str.contains(moa.lower()) & is_DILI

    if moa == 'Hepatitis':
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('inflammation') & is_DILI
        )

    if moa == 'Necrosis':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('cell death') & is_DILI

    if moa == 'Mitochondrial dysfunction':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('mitochond') & is_DILI

    if moa == 'CYP':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('p450') & is_DILI
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('reactive metabolites') & is_DILI
        )

    if moa == 'Oxidative stress':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('oxidative') & is_DILI
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('reactive oxygen species') & is_DILI
        )

    if moa == 'Cholestasis/Biliary':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('cholesta') & is_DILI
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('impaired bile flow') & is_DILI
        )
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('biliary') & is_DILI
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('bile duct') & is_DILI

    if moa == 'Steatosis':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('fatty liver') & is_DILI
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('steato') & is_DILI

    if moa == 'Fibrosis':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('fibrotic') & is_DILI

    if moa == 'Lactic acidosis':
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('metabolic acidosis') & is_DILI
        )

    if moa == 'Hypersensitivity':
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('allergic') & is_DILI
        df_tmp[f'livertox_{moa}'] &= ~livertox_text.str.contains(
            'hypersensitivity syndrome'
        )

    if moa == 'Immune-mediated':
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('immunological') & is_DILI
        )
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('immunoallergic') & is_DILI
        )
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('immunoallergenic') & is_DILI
        )
        df_tmp[f'livertox_{moa}'] |= livertox_text.str.contains('immunogenic') & is_DILI
        df_tmp[f'livertox_{moa}'] |= (
            livertox_text.str.contains('immune response') & is_DILI
        )

    print(moa, df_tmp[f'livertox_{moa}'].sum())

for cmpd in df_tmp.index:
    df_tmp.loc[cmpd, 'livertox_mechanism_summary'] = ', '.join(
        [moa for moa in moas if df_tmp.loc[cmpd, f'livertox_{moa}']]
    )

Hepatitis 268
Necrosis 94
Mitochondrial dysfunction 16
CYP 209
Oxidative stress 8
Cholestasis/Biliary 262
Sinusoidal 31
Steatosis 57
Fibrosis 38
Cirrhosis 54
Lactic acidosis 24
Encephalopathy 27
Immune-mediated 279
Hypersensitivity 341
Hypersensitivity Syndrome 13


In [ ]:
df_DILI['livertox_mechanism_summary'] = (
    df_tmp['livertox_mechanism_summary']
    + np.where(df_DILI['livertox_iDILI'] != '', ', ', '')
    + df_DILI['livertox_iDILI']
)

In [ ]:
manual_annotation = {
    'Bromfenac': 'Necrosis (manual annotation)',
    'Cefuroxime': 'Cholestasis/Biliary (manual annotation)',
    'Citalopram': 'Cholestasis/Biliary, CYP (manual annotation)',
    'Cloxacillin': 'Hepatitis, Cholestasis/Biliary, Hypersensitivity (manual annotation)',
    'Cyclofenil': 'Hepatitis, Cholestasis/Biliary (manual annotation)',
    'Etoposide': 'Hepatitis, Sinusoidal (manual annotation)',
    'Flucloxacillin': 'Hypersensitivity, idiosyncratic (manual annotation)',
    'Glafenine': 'Cirrhosis (manual annotation)',
    'Glimepiride': 'Cholestasis/Biliary (manual annotation)',
    'Glipizide': 'Hepatitis (manual annotation)',
    'Ibufenac': 'Necrosis (manual annotation)',
    'Mefenamic acid': 'Hypersensitivity, idiosyncratic (manual annotation)',
    'Mesalazine': 'Cholestasis/Biliary (manual annotation)',
    'Methazolamide': 'Cirrhosis, Encephalopathy (manual annotation)',
    'Milnacipran': 'Immune-mediated (manual annotation)',
    'Moxisylyte': 'Immune-mediated, Hypersensitivity (manual annotation)',
    'Mycophenolate mofetil': 'Immune-mediated, idiosyncratic (manual annotation)',
    'Phenylbutazone': 'Immune-mediated, idiosyncratic (manual annotation)',
    'Pyrimethamine': 'Cholestasis/Biliary, hypersensitivity (manual annotation)',
    'Sitaxsentan': 'idiosyncratic (manual annotation)',
    'Tenoxicam': 'Cholestasis/Biliary, Immune-mediated (manual annotation)',
    'Ticrynafen': 'Hepatitis, Cirrhosis, Hypersensitivity (manual annotation)',
    'Trimethoprim': 'Cholestasis/Biliary, idiosyncratic (manual annotation)',
    'Trovafloxacin': 'idiosyncratic (manual annotation)',
    'Valproic acid': 'Oxidative stress, Mitochondrial dysfunction, Steatosis (manual annotation)',
    'Ximelagatran': 'Immune-mediated, Hypersensitivity, idiosyncratic (manual annotation)',
}

for cmpd, moa in manual_annotation.items():
    df_DILI.loc[cmpd, 'livertox_mechanism_summary'] = moa

## 7. Summary stats
## 7. 统计摘要

生成DILI标签的交叉表，查看不同数据源之间的对应关系。

In [ ]:
df_DILI['livertox_score'] = df_DILI['livertox_score'].str.replace(' ', '', regex=False)

In [ ]:
df_ct = pd.crosstab(df_DILI['DILIrank'], df_DILI['livertox_score'])
df_ct.sort_values('A', ascending=False)

livertox_score,,A,A[HD],B,B[HD],C,C[HD],D,D[HD],E,E*,X
DILIrank,,,,,,,,,,,,
,661,60,10,41,2,52,5,81,1,374,147,2
Most-DILI-Concern,69,46,2,26,0,33,0,12,0,1,3,0
Less-DILI-Concern,23,21,6,55,2,69,3,67,0,20,12,0
Ambiguous DILI-concern,68,6,2,2,0,5,1,38,1,75,56,0
No-DILI-Concern,148,0,2,1,0,4,0,6,0,130,21,0


In [ ]:
df_ct = pd.crosstab(
    df_DILI['DILIrank'],
    df_DILI['livertox_score'],
    df_DILI['DILI_label_section'],
    aggfunc=set,
)
df_ct.sort_values('A', ascending=False)

livertox_score,,A,A,A [HD],B,B[HD],C,C[HD],D,D[HD],E,E*,X
DILIrank,,,,,,,,,,,,,
,"{, No DILI, Likely, Few cases, Withdrawn}",{Known},{Known},{Known},{Likely},{Likely},{Few cases},{Few cases},"{No DILI, Few cases}",{Few cases},{No DILI},{No DILI},{}
Ambiguous DILI-concern,"{, No DILI}",{Likely},{Likely},{Likely},{Likely},NaN,{Few cases},{Few cases},{Few cases},{Few cases},{No DILI},{No DILI},NaN
Less-DILI-Concern,"{, Few cases}",{Likely},NaN,{Likely},{Likely},{Likely},{Few cases},{Few cases},{Few cases},NaN,{Unlikely},{Unlikely},NaN
Most-DILI-Concern,"{Withdrawn, Likely}","{Withdrawn, Known}",NaN,{Known},"{Withdrawn, Likely}",NaN,"{Withdrawn, Likely}",NaN,{Likely},NaN,{Likely},{Likely},NaN
No-DILI-Concern,{No DILI},NaN,NaN,{Likely},{Likely},NaN,{No DILI},NaN,{No DILI},NaN,{No DILI},{No DILI},NaN


## 8. Cleanup
## 8. 清理

删除重复的化合物记录，确保每个化合物只有一条记录。

In [ ]:
df_DILI[df_DILI.index.str.lower().duplicated()]

,LTKBID,compound_name,DILIrank,label_section,severity_class,DILIst,roa,source,livertox_score,livertox_primary_classification,...,CAS_number,DMSO_mM_solubility,DILI_label,DILI_label_binary,DILI_label_section,livertox_mechanism,livertox_information,livertox_updated,livertox_iDILI,livertox_mechanism_summary
"Estrogens, Conjugated",NaN,"Estrogens, Conjugated",,,NaN,NaN,NaN,NaN,,NaN,...,NaN,NaN,,,,NaN,NaN,NaN,,
esomeprazole Sodium,NaN,esomeprazole Sodium,,,NaN,NaN,NaN,NaN,,NaN,...,161796-78-7,198.69,,,,NaN,NaN,NaN,,
gadolinium ethoxybenzyl DTPA,NaN,gadolinium ethoxybenzyl DTPA,,,NaN,NaN,NaN,NaN,,NaN,...,NaN,NaN,,,,NaN,NaN,NaN,,


In [ ]:
df_DILI.index = capitalize_if_lower(df_DILI.index)
df_DILI = drop_duplicated_index(df_DILI)

In [ ]:
len(df_DILI)

2472

In [ ]:
df_DILI.memory_usage(deep=True).sort_values()

polymer_flag              19776
usan_year                 19776
prodrug                   19776
orphan                    19776
inorganic_flag            19776
                         ...   
Index                    169581
smiles                   233640
information              325230
livertox_mechanism      2800630
livertox_information    4364485
Length: 63, dtype: int64

## 9. Push file to S3
## 9. 将文件推送到S3

将清理后的数据保存到S3存储，供后续步骤使用。

In [ ]:
df_DILI_clean = df_DILI.copy()

for col in ['livertox_mechanism', 'livertox_information']:
    if col in df_DILI_clean.columns:
        df_DILI_clean.pop(col)

In [ ]:
# dmap.s3.write(df_DILI_clean, 'compound_DILI_labels.csv')

Package: s3://dilimap/public/data. Top hash: ff7d660e50


Copying objects: 100%|██████████████████████| 1.22M/1.22M [00:01<00:00, 709kB/s]


Package public/data@d3ef751 pushed to s3://dilimap
Run `quilt3 catalog s3://dilimap/` to browse.
Successfully pushed the new package
